# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from haversine_build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 478

In [3]:
df = pd.read_csv(f"../../../data/top30groups/LongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [6]:
geodata = ['longitude', 'latitude']
combined_geo = df.copy()
combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
combined_geo = combined_geo.drop(columns=geodata)

In [7]:
import ast

def to_tuple_if_needed(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val  # already a tuple

combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [8]:
torch.cuda.empty_cache()


In [9]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
# Hyperparameter grid
param_grid = {
    'lr': [0.001],
    'n_tree': [40, 80],
    'tree_depth': [8, 10],
    'tree_feature_rate': [0.1, 0.3, 0.5],
    'feat_dropout': [0.0, 0.1, 0.2],
    'embed_dim': [64]
    }

# Convert to list of dicts (cartesian product)
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")

    data, edge_index_full, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
        combined_geo, label_index, continuous_col=col)

    best_run = None
    best_score = -1

    for combo in grid_combos:
        args = {
            **dict(zip(param_names, combo)),
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 1500,
            'final_evaluation': False
        }

        print(f"Running config: {args}")
        try:
            acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
                data, edge_index_full, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
                args, row_to_node_index, index_to_label, verbose=False)

            if acc > best_score:
                best_score = acc
                best_run = {
                    "args": args,
                    "acc": acc,
                    "epoch": epoch,
                    "y_pred": y_pred_decoded,
                    "y_true": y_true_decoded,
                    "precision": p,
                    "recall": r,
                    "f1": f1,
                    "micro": (p_micro, r_micro, f1_micro),
                    "macro": (p_macro, r_macro, f1_macro),
                    "auroc": (auc_w, auc_mi, auc_ma),
                    "epoch_logs": epoch_logs
                }
        except Exception as e:
            print(f"Error with config {args}: {e}")
            continue

    # Save best results
    if best_run:
        best_args = best_run["args"]
        os.makedirs(f"Results{partition}", exist_ok=True)

        results_path = f"Results{partition}/Results_{col}_prediction"
        with open(results_path, "w") as f:
            f.write(f"Best acc: {best_run['acc']:.4f} at epoch {best_run['epoch']} for {col} prediction\n")
            f.write(f"Config: {best_args}\n")
            f.write(f"Weighted Precision: {best_run['precision']:.4f}, Recall: {best_run['recall']:.4f}, F1: {best_run['f1']:.4f}\n")
            f.write(f"Macro Precision: {best_run['macro'][0]:.4f}, Recall: {best_run['macro'][1]:.4f}, F1: {best_run['macro'][2]:.4f}\n")
            f.write(f"Micro Precision: {best_run['micro'][0]:.4f}, Recall: {best_run['micro'][1]:.4f}, F1: {best_run['micro'][2]:.4f}\n")
            f.write(f"AUROC Weighted: {best_run['auroc'][0]:.4f}, Micro: {best_run['auroc'][1]:.4f}, Macro: {best_run['auroc'][2]:.4f}\n")

        log_path = f"Results{partition}/epoch_logs_{col}_prediction"
        with open(log_path, "w") as f:
            f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

        y_preds.append(best_run['y_pred'])
        y_trues.append(best_run['y_true'])

print(best_score)


Training model for weaptype1 prediction...
Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Best validation acc: 0.8843 @ epoch 1495
Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Best validation acc: 0.8918 @ epoch 1498
Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Best validation acc: 0.8790 @ epoch 1498
Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Early sto

In [10]:
final_args = best_run['args'].copy()
final_args['epochs'] = 3000
final_args['final_evaluation'] = True

print("\nRunning final evaluation with best config:")
print(final_args)

data, edge_index_full, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
    combined_geo, label_index, continuous_col='weaptype1')

# Run final training and testing
final_acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
    data, edge_index_full, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
    final_args, row_to_node_index, index_to_label, verbose=True)

print(f"\nTest Accuracy: {final_acc:.4f}")


Running final evaluation with best config:
{'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
Epoch 000 | GCN Loss: 1.2195 | NRF Loss: 3.4013 | Joint: 4.6208 | Val Acc: 0.3092
Epoch 050 | GCN Loss: 1.0688 | NRF Loss: 3.0638 | Joint: 4.1326 | Val Acc: 0.6100
Epoch 100 | GCN Loss: 1.0605 | NRF Loss: 2.8736 | Joint: 3.9341 | Val Acc: 0.7674
Epoch 150 | GCN Loss: 1.0567 | NRF Loss: 2.7196 | Joint: 3.7763 | Val Acc: 0.8203
Epoch 200 | GCN Loss: 1.0540 | NRF Loss: 2.5821 | Joint: 3.6361 | Val Acc: 0.8445
Epoch 250 | GCN Loss: 1.0514 | NRF Loss: 2.4569 | Joint: 3.5083 | Val Acc: 0.8609
Epoch 300 | GCN Loss: 1.0488 | NRF Loss: 2.3419 | Joint: 3.3907 | Val Acc: 0.8722
Epoch 350 | GCN Loss: 1.0460 | NRF Loss: 2.2343 | Joint: 3.2803 | Val Acc: 0.8782
Epoch 400 | GCN Loss: 1.0433 | NRF Loss: 2.1337 | Joint: 3.1770 | Val Acc: 0.8867
Epoch 450 | GCN Loss: 1.0406

In [11]:
import os
import json

# Create results directory
os.makedirs(f"Results{partition}", exist_ok=True)

# Save metrics and config
results_path = f"Results{partition}/FinalResults_{continuous_cols[0]}_prediction.txt"
with open(results_path, "w") as f:
    f.write(f"Final Test Accuracy: {final_acc:.4f} at epoch {epoch} for {continuous_cols[0]} prediction\n")
    f.write(f"Final Config: {final_args}\n\n")
    f.write(f"Weighted Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}\n")
    f.write(f"Macro Precision: {p_macro:.4f}, Recall: {r_macro:.4f}, F1: {f1_macro:.4f}\n")
    f.write(f"Micro Precision: {p_micro:.4f}, Recall: {r_micro:.4f}, F1: {f1_micro:.4f}\n")
    f.write(f"AUROC Weighted: {auc_w:.4f}, Micro: {auc_mi:.4f}, Macro: {auc_ma:.4f}\n")


# Save predictions
with open(f"Results{partition}/FinalPredictions_{continuous_cols[0]}.csv", "w") as f:
    f.write("predicted,true\n")
    for pred, true in zip(y_pred_decoded, y_true_decoded):
        f.write(f"{pred},{true}\n")

# Save epoch logs
with open(f"Results{partition}/FinalEpochLogs_{continuous_cols[0]}.txt", "w") as f:
    f.write('\n'.join(f"{x:.4f}" for x in epoch_logs))

print("✅ Final evaluation results saved.")


✅ Final evaluation results saved.


In [12]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [13]:
print(best_run)

{'args': {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}, 'acc': 0.9291797876358032, 'epoch': 1499, 'y_pred': ["Donetsk People's Republic", 'Al-Qaida in the Arabian Peninsula (AQAP)', 'Tupac Amaru Revolutionary Movement (MRTA)', 'National Liberation Army of Colombia (ELN)', "Kurdistan Workers' Party (PKK)", 'Tupac Amaru Revolutionary Movement (MRTA)', "New People's Army (NPA)", 'Palestinians', 'Houthi extremists (Ansar Allah)', 'Tupac Amaru Revolutionary Movement (MRTA)', 'Farabundo Marti National Liberation Front (FMLN)', 'Corsican National Liberation Front (FLNC)', 'Palestinians', 'Taliban', 'National Liberation Army of Colombia (ELN)', "New People's Army (NPA)", 'Palestinians', 'Nicaraguan Democratic Force (FDN)', 'Al-Shabaab', 'Irish Republican Army (IRA)', 'Al-Qaida in Iraq', 'Sikh Extremists', 'Irish Republican Army (IRA)', 'Al-Shabaab', '

In [14]:
print(best_run['args'])

{'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}


In [15]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [16]:
best_score

0.9291797876358032

In [17]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

'\nBest acc: 0.9205 at epoch 750 for weaptype1 prediction\nWeighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191\nMacro Precision: 0.9176, Recall: 0.9107, F1: 0.9105\nMicro Precision: 0.9205, Recall: 0.9205, F1: 0.9205\nAUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963\n\n'

In [18]:
from sklearn.metrics import classification_report

print(classification_report(y_preds[-1], y_trues[-1]))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       1.00      0.96      0.98       263
        African National Congress (South Africa)       1.00      1.00      1.00       313
                                Al-Qaida in Iraq       0.87      0.77      0.82       457
        Al-Qaida in the Arabian Peninsula (AQAP)       0.94      0.90      0.92       338
                                      Al-Shabaab       1.00      0.99      0.99       333
             Basque Fatherland and Freedom (ETA)       1.00      1.00      1.00       321
                                      Boko Haram       0.97      0.98      0.98       228
  Communist Party of India - Maoist (CPI-Maoist)       0.95      0.85      0.90       198
       Corsican National Liberation Front (FLNC)       1.00      0.99      1.00       405
                       Donetsk People's Republic       1.00      1.00      1.00       359
Farabundo

In [19]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [20]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 478 to Results478/cm_478_weaptype1.png
